# `xp_20260309`

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Set configuration parameters

In [ ]:
MODEL_NAME = "google/flan-t5-small"
# The context window of the base model
N_CONTEXT_WINDOW = 512
# The maximum number of words in each input (based on the assumption that 1 token ~= 0.75 words)
MAX_N_INPUT_TOKENS =  round(N_CONTEXT_WINDOW * 0.70)
MAX_N_INPUT_WORDS = round(MAX_N_INPUT_TOKENS * 0.75)
# The maximum number of words in each label (based on the assumption that 1 token ~= 0.75 words)
MAX_N_LABEL_TOKENS = round(N_CONTEXT_WINDOW * 0.20)
MAX_N_LABEL_WORDS = round(MAX_N_LABEL_TOKENS * 0.75)

## Create pre-trained base model instance

Check `~/.cache/huggingface/`.

In [ ]:
# Select CUDA device index
import os
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, quantization_config=BitsAndBytesConfig(load_in_8bit=True))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

## Create pre-trained PEFT model instance

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType


def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"{'trainable_params'.ljust(16)}: {trainable_params}"
    )
    print(
        f"{'all params'.ljust(16)}: {all_param}"
    )
    print(
        f"{'trainable %'.ljust(16)}: {100 * trainable_params / all_param}"
    )


lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q", "v"], lora_dropout=0.05, bias="none", task_type="SEQ_2_SEQ_LM"
)


model = get_peft_model(model, lora_config)
print_trainable_parameters(model)

## Load datasets

In [ ]:
dataset = load_dataset('RealTimeData/bbc_news_alltime', data_dir='2020-02')

Only use the examples that can fit within the context window of the model with enough buffer room for the model to generate its output.

In [ ]:
dataset = dataset.filter(lambda x: len(x["content"].split()) <= MAX_N_INPUT_WORDS)

In [ ]:
dataset.keys()

In [ ]:
dataset["train"]

In [ ]:
dataset = dataset["train"].train_test_split(test_size=0.1)
train_val_dataset = dataset["train"].train_test_split(test_size=0.1)

dataset["train"] = train_val_dataset["train"]
dataset["validation"] = train_val_dataset["test"]
# dataset["test"] has already been created

In [ ]:
dataset.keys()

In [ ]:
dataset["train"]

In [ ]:
dataset["validation"]

## Pre-process and tokenize datasets

In [ ]:
tokenizer.chat_template = open("news-headline-0.1-template.jinja").read()

In [ ]:
def generate_chat_messages(input_text: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": "Generate a concise news headline."},
        {"role": "user", "content": f'Article:\n"""\n{input_text}\n"""'},
    ]

In [ ]:
input_text = dataset["train"]["content"][0]
messages = generate_chat_messages(input_text=input_text)
formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(formatted_text)

In [ ]:
def preprocess_batch(examples, input_column, label_column, max_length):    
    inputs = examples[input_column]
    labels = examples[label_column]

    message_groups = [generate_chat_messages(input_text=input_text) for input_text in inputs]
    formatted_inputs = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) for messages in message_groups]
    
    model_inputs = tokenizer(
        formatted_inputs,
        # The maximum length of each input text sequence
        max_length=max_length,
        # Pad sequences shorter than the max length up to the max length
        padding="max_length", 
        # Cut off sequences longer than the max length
        truncation=True, 
        # Return token IDs as PyTorch tensors
        return_tensors="pt"
    )
    labels = tokenizer(
        labels, 
        max_length=max_length, 
        padding="max_length", 
        truncation=True, 
        return_tensors="pt"
    )

    # Replace all padding token IDs with '-100'
    # The tokens corresponding to pad_token_id needs to be set to -100 so that the CrossEntropy loss associated with the model will correctly ignore these tokens.
    labels[labels["input_ids"] == tokenizer.pad_token_id] = -100
    # Add 'labels' as an additional key to the `model_inputs` dict
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

def preprocess_dataset(dataset):
    processed_datasets = dataset.map(
        preprocess_batch,
        batched=True,
        num_proc=1,
        remove_columns=dataset["train"].column_names,
        load_from_cache_file=False,
        desc="Running tokenizer on dataset",
        fn_kwargs={
            "input_column": "content",
            "label_column": "title", 
            "max_length": MAX_N_INPUT_TOKENS,
        },
    )
    
    train_dataset = processed_datasets["train"]
    val_dataset = processed_datasets["validation"]

    return train_dataset, val_dataset

train_dataset, val_dataset = preprocess_dataset(dataset=dataset)

## Fine-tune the PEFT model

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="trainer_output",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    learning_rate=1e-3,
    # Set evaluations to run at the end of every epoch
    eval_strategy="epoch",
    # effective_batch_size = train_batch_size × gradient_accumulation_steps
    gradient_accumulation_steps=1,
    # Automatically reduces the batch size if you hit out-of-memory (OOM) errors
    auto_find_batch_size=True,
    # Save a checkpoint every 100 training steps
    # If more than 8 exist, oldest checkpoints are deleted
    save_steps=100,
    save_total_limit=8,
    # Log the training loss every 1 training steps
    logging_strategy="steps",
    logging_steps=1,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)
model.config.use_cache = False  # Silence the warnings. Please re-enable for inference!

In [ ]:
# TODO: Use appropriate loss function / evaluation metric

In [ ]:
# TODO: Plot training / validation loss graph

In [ ]:
trainer.train()

In [ ]:
model.eval()

input_text = dataset["validation"]["content"][1]
input_messages = generate_chat_messages(input_text=input_text)
input_text = tokenizer.apply_chat_template(input_messages, tokenize=False, add_generation_prompt=True)
input_tokens = tokenizer(formatted_text, return_tensors="pt").to(model.device)

output_tokens = model.generate(input_ids=input_tokens["input_ids"], max_new_tokens=MAX_N_LABEL_TOKENS)
output_text = tokenizer.batch_decode(output_tokens.detach().cpu().numpy(), skip_special_tokens=True)

print(f"INPUT TEXT: \n{input_text}")
print()
print(f"OUTPUT TEXT:\n{output_text}")

## Share your adapters on 🤗 Hub

Once you have trained your adapter, you can easily share it on the Hub using the method `push_to_hub` . Note that only the adapter weights and config will be pushed.

In [ ]:
# from huggingface_hub import notebook_login

# notebook_login()

In [ ]:
# model.push_to_hub("jyjulianwong/flan-t5-small-news-headline-0.1", use_auth_token=True)

## Load your adapter from the Hub

You can load the model together with the adapter with few lines of code! Check the snippet below to load the adapter from the Hub and run the example evaluation!

In [ ]:
# import torch
# from peft import PeftModel, PeftConfig
# from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# peft_model_id = "jyjulianwong/flan-t5-small-news-headline-0.1"
# config = PeftConfig.from_pretrained(peft_model_id)

# model = AutoModelForSeq2SeqLM.from_pretrained(config.base_model_name_or_path, dtype="auto", device_map="auto")
# tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

# # Load the PEFT model
# model = PeftModel.from_pretrained(model, peft_model_id)

In [ ]:
# model.eval()

# input_text = dataset["validation"]["content"][0]
# input_messages = generate_chat_messages(input_text=input_text)
# input_text = tokenizer.apply_chat_template(input_messages, tokenize=False, add_generation_prompt=True)
# input_tokens = tokenizer(formatted_text, return_tensors="pt").to(model.device)

# output_tokens = model.generate(input_ids=input_tokens["input_ids"], max_new_tokens=MAX_N_LABEL_TOKENS)
# output_text = tokenizer.batch_decode(output_tokens.detach().cpu().numpy(), skip_special_tokens=True)

# print(f"INPUT TEXT: \n{input_text}")
# print()
# print(f"OUTPUT TEXT:\n{output_text}")